# TimesFM

# Setup

In [ ]:
!pip install timesfm mlflow --quiet
import pandas as pd, numpy as np, mlflow, os
import matplotlib.pyplot as plt
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
dagshub_token = user_secrets.get_secret("DAGSHUB_TOKEN")
os.environ['MLFLOW_TRACKING_USERNAME'] = 'gdzag22'
os.environ['MLFLOW_TRACKING_PASSWORD'] = dagshub_token
mlflow.set_tracking_uri("https://dagshub.com/Nestor-Dzadzamia/walmart-sales-forecasting.mlflow")
mlflow.set_experiment("TimesFM_Training")

# Shared functions (from EDA notebook)

In [ ]:
def load_and_merge(df, features, stores):
    out = df.merge(features, on=['Store', 'Date', 'IsHoliday'], how='left')
    out = out.merge(stores, on='Store', how='left')
    out['Date'] = pd.to_datetime(out['Date'])
    return out.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

MD_COLS = [f"MarkDown{i}" for i in range(1, 6)]

def preprocess(df):
    out = df.copy()
    out[MD_COLS] = out[MD_COLS].fillna(0)
    out[["CPI", "Unemployment"]] = out.groupby("Store")[["CPI", "Unemployment"]].ffill()
    return out

def wmae(y_true, y_pred, is_holiday):
    w = np.where(is_holiday, 5, 1)
    return np.sum(w * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(w)

def coldstart_fallback(df_train, df_test):
    train_pairs = set(zip(df_train.Store, df_train.Dept))
    mask = ~pd.Series(list(zip(df_test.Store, df_test.Dept)), index=df_test.index).isin(train_pairs)
    cold = df_test[mask].copy()
    recent = df_train[df_train['Date'] >= df_train['Date'].max() - pd.Timedelta(weeks=52)]
    med = recent.groupby(['Type', 'Dept'])['Weekly_Sales'].median()
    cold['y_fallback'] = [med.get((t, d), 0.0) for t, d in zip(cold['Type'], cold['Dept'])]
    cold['y_fallback'] = cold['y_fallback'].clip(lower=0)
    return cold

# Load and Merge Data

In [ ]:
path = "/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/"
train = pd.read_csv(path + "train.csv.zip")
test = pd.read_csv(path + "test.csv.zip")
features = pd.read_csv(path + "features.csv.zip")
stores = pd.read_csv(path + "stores.csv")

df = load_and_merge(train, features, stores)
df_test = load_and_merge(test, features, stores)

print("train merged:", df.shape)
print("test merged:", df_test.shape)

# Cleaning

In [ ]:
with mlflow.start_run(run_name="TimesFM_Cleaning"):
    mlflow.log_param("markdown_fill", "0")
    mlflow.log_param("cpi_unemployment_fill", "ffill_per_store")
    mlflow.log_metric("train_rows_before", len(df))
    mlflow.log_metric("markdown_nan_before", int(df[MD_COLS].isna().sum().sum()))

    df_clean = preprocess(df)

    mlflow.log_metric("train_rows_after", len(df_clean))
    mlflow.log_metric("markdown_nan_after", int(df_clean[MD_COLS].isna().sum().sum()))

print(df_clean.shape)
print("Remaining MarkDown NaNs:", df_clean[MD_COLS].isna().sum().sum())

# Validation Split

In [ ]:
VALIDATION_START = df_test['Date'].min() - pd.Timedelta(weeks=52)
VALIDATION_END = VALIDATION_START + pd.Timedelta(weeks=39)

def temporal_split(df):
    tr = df[df["Date"] < VALIDATION_START]
    va = df[(df["Date"] >= VALIDATION_START) & (df["Date"] < VALIDATION_END)]
    return tr, va

tr, va = temporal_split(df_clean)
print("train:", tr["Date"].max(), tr.shape)
print("val:", va["Date"].min(), va.shape)

In [ ]:
import timesfm
print(timesfm.__file__)
print(dir(timesfm))

In [ ]:
model_class = timesfm.TimesFM_2p5_200M_torch
print("Class methods:", [m for m in dir(model_class) if not m.startswith('_')])
print()
print(model_class.__doc__)
print()
print("ForecastConfig fields:")
import dataclasses
for f in dataclasses.fields(timesfm.ForecastConfig):
    print(f"  {f.name}: default={f.default}")

In [ ]:
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")

model.compile(
    timesfm.ForecastConfig(
        max_context=104,
        max_horizon=39,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)

# Zero-Shot Forecast: 5 Representative Series

In [ ]:
examples = [(1, 1), (4, 92), (20, 1), (10, 30), (33, 40)]

timesfm_results = {}

for s, d in examples:
    series = df_clean[(df_clean['Store'] == s) & (df_clean['Dept'] == d)].sort_values('Date')
    y = series['Weekly_Sales'].values
    is_holiday = series['IsHoliday'].values[-39:]

    train_y = y[:-39]
    test_y = y[-39:]

    with mlflow.start_run(run_name=f"TimesFM_ZeroShot_Store{s}_Dept{d}"):
        mlflow.log_param("max_context", 104)
        mlflow.log_param("max_horizon", 39)
        mlflow.log_param("mode", "zero-shot")

        point_forecast, quantile_forecast = model.forecast(horizon=39, inputs=[train_y])
        pred = point_forecast[0]

        score = wmae(test_y, pred, is_holiday)
        mlflow.log_metric("wmae", score)

    timesfm_results[f"Store{s}_Dept{d}"] = {'actual': test_y, 'forecast': pred, 'wmae': score}
    print(f"Store {s}, Dept {d}: TimesFM zero-shot WMAE = {score:.2f}")

In [ ]:
fig, axes = plt.subplots(len(examples), 1, figsize=(12, 12))
for ax, (s, d) in zip(axes, examples):
    key = f"Store{s}_Dept{d}"
    r = timesfm_results[key]
    ax.plot(r['actual'], label='Actual')
    ax.plot(r['forecast'], label='TimesFM Zero-Shot')
    ax.set_title(f"Store {s}, Dept {d} — WMAE {r['wmae']:.0f}")
    ax.legend()
plt.tight_layout()
plt.show()

# Context Length Sweep

In [ ]:
context_lengths = [26, 52, 104]
context_results = {}

for ctx in context_lengths:
    model_ctx = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
    model_ctx.compile(timesfm.ForecastConfig(
        max_context=ctx, max_horizon=39, normalize_inputs=True,
        use_continuous_quantile_head=True, force_flip_invariance=True,
        infer_is_positive=True, fix_quantile_crossing=True,
    ))

    scores = []
    for s, d in examples:
        series = df_clean[(df_clean['Store'] == s) & (df_clean['Dept'] == d)].sort_values('Date')
        y = series['Weekly_Sales'].values
        is_holiday = series['IsHoliday'].values[-39:]
        train_y = y[:-39][-ctx:]
        test_y = y[-39:]

        point_forecast, _ = model_ctx.forecast(horizon=39, inputs=[train_y])
        scores.append(wmae(test_y, point_forecast[0], is_holiday))

    avg = np.mean(scores)
    context_results[ctx] = avg

    with mlflow.start_run(run_name=f"TimesFM_Context_{ctx}"):
        mlflow.log_param("max_context", ctx)
        mlflow.log_metric("avg_wmae", avg)

    print(f"max_context={ctx}: avg WMAE = {avg:.2f}")

In [ ]:
print("Model's max supported context:", model.forecast_config.max_context if hasattr(model, 'forecast_config') else "check model card")

In [ ]:
context_lengths_extended = [104, 128]
context_results_ext = dict(context_results)

for ctx in context_lengths_extended[1:]:
    model_ctx = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
    model_ctx.compile(timesfm.ForecastConfig(
        max_context=ctx, max_horizon=39, normalize_inputs=True,
        use_continuous_quantile_head=True, force_flip_invariance=True,
        infer_is_positive=True, fix_quantile_crossing=True,
    ))

    scores = []
    for s, d in examples:
        series = df_clean[(df_clean['Store'] == s) & (df_clean['Dept'] == d)].sort_values('Date')
        y = series['Weekly_Sales'].values
        is_holiday = series['IsHoliday'].values[-39:]
        train_y = y[:-39][-ctx:]
        test_y = y[-39:]

        point_forecast, _ = model_ctx.forecast(horizon=39, inputs=[train_y])
        scores.append(wmae(test_y, point_forecast[0], is_holiday))

    avg = np.mean(scores)
    context_results_ext[ctx] = avg

    with mlflow.start_run(run_name=f"TimesFM_Context_{ctx}"):
        mlflow.log_param("max_context", ctx)
        mlflow.log_metric("avg_wmae", avg)

    print(f"max_context={ctx}: avg WMAE = {avg:.2f}")

print()
print("Full sweep:", context_results_ext)

In [ ]:
best_timesfm_context = 104

In [ ]:
help(model.forecast_with_covariates)

In [ ]:
model.compile(timesfm.ForecastConfig(
    max_context=104, max_horizon=39, normalize_inputs=True,
    use_continuous_quantile_head=True, force_flip_invariance=True,
    infer_is_positive=True, fix_quantile_crossing=True,
    return_backcast=True,
))

In [ ]:

def get_holiday_covariate(store, dept, context_len):
    hist = df_clean[(df_clean['Store']==store) & (df_clean['Dept']==dept)].sort_values('Date')
    hist_context = hist.iloc[:-39].tail(context_len)
    future_dates = df_clean[(df_clean['Store']==store) & (df_clean['Dept']==dept)].sort_values('Date').iloc[-39:]
    combined_holiday = pd.concat([hist_context['IsHoliday'], future_dates['IsHoliday']]).astype(float).tolist()
    return combined_holiday

inputs_list = []
holiday_covariates = []
static_types = []
actuals_list = []
holidays_test_list = []

for s, d in examples:
    series = df_clean[(df_clean['Store'] == s) & (df_clean['Dept'] == d)].sort_values('Date')
    y = series['Weekly_Sales'].values
    train_y = y[:-39][-104:]
    test_y = y[-39:]

    inputs_list.append(train_y)
    holiday_covariates.append(get_holiday_covariate(s, d, 104))
    static_types.append(series['Type'].iloc[0])
    actuals_list.append(test_y)
    holidays_test_list.append(series['IsHoliday'].values[-39:])

outputs, xreg_outputs = model.forecast_with_covariates(
    inputs=inputs_list,
    dynamic_numerical_covariates={'is_holiday': holiday_covariates},
    static_categorical_covariates={'store_type': static_types},
    xreg_mode="xreg + timesfm",
)

scores_cov = []
for i, (s, d) in enumerate(examples):
    pred = outputs[i][-39:] if len(outputs[i]) > 39 else outputs[i]
    score = wmae(actuals_list[i], pred, holidays_test_list[i])
    scores_cov.append(score)
    print(f"Store {s}, Dept {d}: with covariates WMAE = {score:.2f}")

avg_cov = np.mean(scores_cov)
print(f"\nAverage with covariates: {avg_cov:.2f}")
print(f"Average without covariates (context=104): 3960.43")

with mlflow.start_run(run_name="TimesFM_Covariates_IsHoliday_StoreType"):
    mlflow.log_param("covariates", "is_holiday (dynamic), store_type (static)")
    mlflow.log_param("xreg_mode", "xreg + timesfm")
    mlflow.log_metric("avg_wmae", avg_cov)
    mlflow.log_metric("avg_wmae_no_covariates", 3960.43)

In [ ]:
model_nonorm = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
model_nonorm.compile(timesfm.ForecastConfig(
    max_context=104, max_horizon=39, normalize_inputs=False,
    use_continuous_quantile_head=True, force_flip_invariance=True,
    infer_is_positive=True, fix_quantile_crossing=True,
))

scores_nonorm = []
for s, d in examples:
    series = df_clean[(df_clean['Store'] == s) & (df_clean['Dept'] == d)].sort_values('Date')
    y = series['Weekly_Sales'].values
    is_holiday = series['IsHoliday'].values[-39:]
    train_y = y[:-39][-104:]
    test_y = y[-39:]

    point_forecast, _ = model_nonorm.forecast(horizon=39, inputs=[train_y])
    scores_nonorm.append(wmae(test_y, point_forecast[0], is_holiday))

avg_nonorm = np.mean(scores_nonorm)
print(f"normalize_inputs=False: avg WMAE = {avg_nonorm:.2f}")
print(f"normalize_inputs=True (current best): 3960.43")

with mlflow.start_run(run_name="TimesFM_NoNormalize"):
    mlflow.log_param("normalize_inputs", False)
    mlflow.log_metric("avg_wmae", avg_nonorm)

In [ ]:
window_results = {}
for ws in [0, 4, 8]:
    model_ws = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
    model_ws.compile(timesfm.ForecastConfig(
        max_context=104, max_horizon=39, normalize_inputs=True, window_size=ws,
        use_continuous_quantile_head=True, force_flip_invariance=True,
        infer_is_positive=True, fix_quantile_crossing=True,
    ))

    scores_ws = []
    for s, d in examples:
        series = df_clean[(df_clean['Store'] == s) & (df_clean['Dept'] == d)].sort_values('Date')
        y = series['Weekly_Sales'].values
        is_holiday = series['IsHoliday'].values[-39:]
        train_y = y[:-39][-104:]
        test_y = y[-39:]
        point_forecast, _ = model_ws.forecast(horizon=39, inputs=[train_y])
        scores_ws.append(wmae(test_y, point_forecast[0], is_holiday))

    avg_ws = np.mean(scores_ws)
    window_results[ws] = avg_ws
    with mlflow.start_run(run_name=f"TimesFM_WindowSize_{ws}"):
        mlflow.log_param("window_size", ws)
        mlflow.log_metric("avg_wmae", avg_ws)
    print(f"window_size={ws}: avg WMAE = {avg_ws:.2f}")

In [ ]:
print("model.forecast_config.normalize_inputs:", model.forecast_config.normalize_inputs)
print("model_nonorm.forecast_config.normalize_inputs:", model_nonorm.forecast_config.normalize_inputs)
print()
print("model_ws (window_size=8) actual config:")
model_ws_check = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
model_ws_check.compile(timesfm.ForecastConfig(
    max_context=104, max_horizon=39, normalize_inputs=True, window_size=8,
    use_continuous_quantile_head=True, force_flip_invariance=True,
    infer_is_positive=True, fix_quantile_crossing=True,
))
print("window_size:", model_ws_check.forecast_config.window_size)

In [ ]:
s, d = 33, 40
series = df_clean[(df_clean['Store'] == s) & (df_clean['Dept'] == d)].sort_values('Date')
y = series['Weekly_Sales'].values
train_y = y[:-39][-104:]

pred_norm_true, _ = model.forecast(horizon=39, inputs=[train_y])
pred_norm_false, _ = model_nonorm.forecast(horizon=39, inputs=[train_y])

print("Max abs difference between normalize=True and normalize=False predictions:", np.max(np.abs(pred_norm_true[0] - pred_norm_false[0])))

In [ ]:
model_true = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
model_true.compile(timesfm.ForecastConfig(
    max_context=104, max_horizon=39, normalize_inputs=True,
    use_continuous_quantile_head=True, force_flip_invariance=True,
    infer_is_positive=True, fix_quantile_crossing=True,
))

model_false = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
model_false.compile(timesfm.ForecastConfig(
    max_context=104, max_horizon=39, normalize_inputs=False,
    use_continuous_quantile_head=True, force_flip_invariance=True,
    infer_is_positive=True, fix_quantile_crossing=True,
))

s, d = 33, 40
series = df_clean[(df_clean['Store'] == s) & (df_clean['Dept'] == d)].sort_values('Date')
y = series['Weekly_Sales'].values
train_y = y[:-39][-104:]

pred_true, _ = model_true.forecast(horizon=39, inputs=[train_y])
pred_false, _ = model_false.forecast(horizon=39, inputs=[train_y])

print("Shapes:", pred_true[0].shape, pred_false[0].shape)
print("Max abs difference:", np.max(np.abs(pred_true[0] - pred_false[0])))

In [ ]:
def blend_holiday_predictions(preds_df, history_df, holiday_flags_df, alpha=0.5):
    out = preds_df.merge(holiday_flags_df[['Store','Dept','Date','IsHoliday']], on=['Store','Dept','Date'], how='left')
    lookup_df = out[['Store','Dept','Date']].copy()
    lookup_df['lookup_date'] = lookup_df['Date'] - pd.Timedelta(weeks=52)
    merged = lookup_df.merge(history_df[['Store','Dept','Date','Weekly_Sales']].rename(
        columns={'Date':'lookup_date','Weekly_Sales':'seasonal_naive'}), on=['Store','Dept','lookup_date'], how='left')
    out['seasonal_naive'] = merged['seasonal_naive'].values
    mask = out['IsHoliday'].fillna(False) & out['seasonal_naive'].notna()
    out.loc[mask, 'Weekly_Sales'] = alpha * out.loc[mask, 'seasonal_naive'] + (1 - alpha) * out.loc[mask, 'Weekly_Sales']
    return out.drop(columns=['IsHoliday', 'seasonal_naive'])

blend_results_timesfm = {}
for alpha in [0.0, 0.3, 0.45, 0.5, 0.55, 0.7, 1.0]:
    scores_b = []
    for i, (s, d) in enumerate(examples):
        series = df_clean[(df_clean['Store'] == s) & (df_clean['Dept'] == d)].sort_values('Date')
        dates_test = series['Date'].values[-39:]
        is_holiday_arr = series['IsHoliday'].values[-39:]

        preds_df = pd.DataFrame({'Store': s, 'Dept': d, 'Date': dates_test, 'Weekly_Sales': outputs[i][-39:]})
        holiday_flags = pd.DataFrame({'Store': s, 'Dept': d, 'Date': dates_test, 'IsHoliday': is_holiday_arr})

        blended = blend_holiday_predictions(preds_df, tr, holiday_flags, alpha=alpha)
        actual = series['Weekly_Sales'].values[-39:]
        scores_b.append(wmae(actual, blended['Weekly_Sales'].values, is_holiday_arr))

    avg_b = np.mean(scores_b)
    blend_results_timesfm[alpha] = avg_b
    print(f"alpha={alpha}: avg WMAE = {avg_b:.2f}")

# Forecast Uncertainty (Quantile Calibration)

In [ ]:
s, d = 33, 40
series = df_clean[(df_clean['Store'] == s) & (df_clean['Dept'] == d)].sort_values('Date')
y = series['Weekly_Sales'].values
train_y = y[:-39][-104:]
test_y = y[-39:]

point_forecast, quantile_forecast = model_true.forecast(horizon=39, inputs=[train_y])

print("Quantile forecast shape:", quantile_forecast.shape)

In [ ]:
quantiles = quantile_forecast[0]  
q10 = quantiles[:, 1]
q90 = quantiles[:, 9]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test_y, label='Actual', color='black')
ax.plot(point_forecast[0], label='TimesFM Forecast', color='purple')
ax.fill_between(range(39), q10, q90, color='purple', alpha=0.2, label='80% Interval (q10-q90)')
ax.set_title(f"Store {s}, Dept {d} — TimesFM Forecast with Quantile Bounds")
ax.legend()
plt.tight_layout()
plt.show()

in_interval = ((test_y >= q10) & (test_y <= q90)).mean()
print(f"Actual values falling inside the 80% interval: {in_interval:.1%} (ideally close to 80%)")
print(f"SARIMA's 95% coverage on the same series was: 84.6%")
print(f"Prophet's 95% coverage on the same series was: 79.5%")

In [ ]:
print("Model class attributes (including private):", [a for a in dir(model_true) if 'model' in a.lower() or 'train' in a.lower() or 'param' in a.lower()])
print()
print("Underlying torch module type:", type(model_true).__mro__)

In [ ]:
print(type(model_true.model))
print()
n_params = sum(p.numel() for p in model_true.model.parameters())
n_trainable = sum(p.numel() for p in model_true.model.parameters() if p.requires_grad)
print(f"Total parameters: {n_params:,}")
print(f"Trainable parameters: {n_trainable:,}")

In [ ]:
for param in model_true.model.parameters():
    param.requires_grad = False

layer_names = [name for name, _ in model_true.model.named_parameters()]
print("Sample of layer names (first 10):", layer_names[:10])
print("Sample of layer names (last 10):", layer_names[-10:])

In [ ]:
for name, param in model_true.model.named_parameters():
    if 'output_projection_point' in name:
        param.requires_grad = True

n_trainable_now = sum(p.numel() for p in model_true.model.parameters() if p.requires_grad)
print(f"Trainable parameters now: {n_trainable_now:,} (out of {n_params:,} total)")

import torch
optimizer = torch.optim.Adam(
    [p for p in model_true.model.parameters() if p.requires_grad],
    lr=1e-5
)

In [ ]:
import inspect
print(inspect.signature(model_true.model.forward))
print()
print(model_true.model.forward.__doc__)

# Full-Scale Validation: All Series, Winning Config

In [ ]:
import time

all_pairs_tfm = list(df_clean.groupby(['Store','Dept']).groups.keys())
CONTEXT_LEN = 104

valid_pairs = []
inputs_batch = []
holiday_cov_batch = []
static_type_batch = []
actuals_batch = []
holiday_test_batch = []
dates_batch = []

for s, d in all_pairs_tfm:
    series = df_clean[(df_clean['Store'] == s) & (df_clean['Dept'] == d)].sort_values('Date')
    y = series['Weekly_Sales'].values
    if len(y) < 39 + 10:  # need at least some context plus the test horizon
        continue

    train_y = y[:-39][-CONTEXT_LEN:]
    test_y = y[-39:]
    if len(train_y) < 10:
        continue

    hist_holiday = series['IsHoliday'].iloc[:-39].tail(len(train_y)).astype(float).tolist()
    future_holiday = series['IsHoliday'].iloc[-39:].astype(float).tolist()

    valid_pairs.append((s, d))
    inputs_batch.append(train_y)
    holiday_cov_batch.append(hist_holiday + future_holiday)
    static_type_batch.append(series['Type'].iloc[0])
    actuals_batch.append(test_y)
    holiday_test_batch.append(series['IsHoliday'].values[-39:])
    dates_batch.append(series['Date'].values[-39:])

print(f"Valid series for validation: {len(valid_pairs)} / {len(all_pairs_tfm)}")

In [ ]:
model_true.compile(timesfm.ForecastConfig(
    max_context=104, max_horizon=39, normalize_inputs=True,
    use_continuous_quantile_head=True, force_flip_invariance=True,
    infer_is_positive=True, fix_quantile_crossing=True,
    return_backcast=True,
))

In [ ]:
BATCH_SIZE = 200
all_outputs = []
start = time.time()

for i in range(0, len(inputs_batch), BATCH_SIZE):
    chunk_inputs = inputs_batch[i:i+BATCH_SIZE]
    chunk_holiday = holiday_cov_batch[i:i+BATCH_SIZE]
    chunk_static = static_type_batch[i:i+BATCH_SIZE]

    outputs_chunk, _ = model_true.forecast_with_covariates(
        inputs=chunk_inputs,
        dynamic_numerical_covariates={'is_holiday': chunk_holiday},
        static_categorical_covariates={'store_type': chunk_static},
        xreg_mode="xreg + timesfm",
    )
    all_outputs.extend(outputs_chunk)
    if (i // BATCH_SIZE) % 5 == 0:
        print(f"  {i + len(chunk_inputs)}/{len(inputs_batch)} done...")

elapsed = time.time() - start
print(f"Total time: {elapsed/60:.1f} minutes")

In [ ]:
results_tfm = []
for i, (s, d) in enumerate(valid_pairs):
    pred = all_outputs[i][-39:] if len(all_outputs[i]) > 39 else all_outputs[i]
    for dt, actual_val, pred_val, is_hol in zip(dates_batch[i], actuals_batch[i], pred, holiday_test_batch[i]):
        results_tfm.append({'Store': s, 'Dept': d, 'Date': dt, 'Weekly_Sales': pred_val,
                             'Actual': actual_val, 'IsHoliday': is_hol})

results_tfm_df = pd.DataFrame(results_tfm)
results_tfm_df['Date'] = pd.to_datetime(results_tfm_df['Date'])

overall_wmae_tfm = wmae(results_tfm_df['Actual'], results_tfm_df['Weekly_Sales'], results_tfm_df['IsHoliday'])
print(f"Full-scale WMAE (covariates, no blend): {overall_wmae_tfm:.2f}")

blended_tfm = blend_holiday_predictions(
    results_tfm_df[['Store','Dept','Date','Weekly_Sales']].copy(),
    tr,
    results_tfm_df[['Store','Dept','Date','IsHoliday']],
    alpha=0.55
)
merged_final = blended_tfm.merge(results_tfm_df[['Store','Dept','Date','Actual','IsHoliday']], on=['Store','Dept','Date'])
overall_wmae_tfm_blended = wmae(merged_final['Actual'], merged_final['Weekly_Sales'], merged_final['IsHoliday'])
print(f"Full-scale WMAE (covariates + blend α=0.55): {overall_wmae_tfm_blended:.2f}")

with mlflow.start_run(run_name="TimesFM_FullScale_Validation"):
    mlflow.log_param("context", 104)
    mlflow.log_param("covariates", "is_holiday, store_type")
    mlflow.log_param("blend_alpha", 0.55)
    mlflow.log_metric("wmae_no_blend", overall_wmae_tfm)
    mlflow.log_metric("wmae_with_blend", overall_wmae_tfm_blended)
    mlflow.log_metric("series_covered", len(valid_pairs))
    mlflow.log_metric("series_total", len(all_pairs_tfm))

# Validation Reliability Check (Second Window)

In [ ]:
VALIDATION_START_2 = pd.Timestamp("2011-05-06")
VALIDATION_END_2 = VALIDATION_START_2 + pd.Timedelta(weeks=39)

valid_pairs_w2 = []
inputs_batch_w2 = []
holiday_cov_batch_w2 = []
static_type_batch_w2 = []
actuals_batch_w2 = []
holiday_test_batch_w2 = []
dates_batch_w2 = []

for s, d in all_pairs_tfm:
    series = df_clean[(df_clean['Store'] == s) & (df_clean['Dept'] == d)].sort_values('Date')
    train_part = series[series['Date'] < VALIDATION_START_2]
    test_part = series[(series['Date'] >= VALIDATION_START_2) & (series['Date'] < VALIDATION_END_2)]

    if len(train_part) < 10 or len(test_part) == 0:
        continue

    train_y = train_part['Weekly_Sales'].values[-104:]
    test_y = test_part['Weekly_Sales'].values

    hist_holiday = train_part['IsHoliday'].tail(len(train_y)).astype(float).tolist()
    future_holiday = test_part['IsHoliday'].astype(float).tolist()

    valid_pairs_w2.append((s, d))
    inputs_batch_w2.append(train_y)
    holiday_cov_batch_w2.append(hist_holiday + future_holiday)
    static_type_batch_w2.append(train_part['Type'].iloc[0])
    actuals_batch_w2.append(test_y)
    holiday_test_batch_w2.append(test_part['IsHoliday'].values)
    dates_batch_w2.append(test_part['Date'].values)

print(f"Valid series for window 2: {len(valid_pairs_w2)} / {len(all_pairs_tfm)}")

In [ ]:
all_outputs_w2 = []
start = time.time()

for i in range(0, len(inputs_batch_w2), BATCH_SIZE):
    chunk_inputs = inputs_batch_w2[i:i+BATCH_SIZE]
    chunk_holiday = holiday_cov_batch_w2[i:i+BATCH_SIZE]
    chunk_static = static_type_batch_w2[i:i+BATCH_SIZE]

    outputs_chunk, _ = model_true.forecast_with_covariates(
        inputs=chunk_inputs,
        dynamic_numerical_covariates={'is_holiday': chunk_holiday},
        static_categorical_covariates={'store_type': chunk_static},
        xreg_mode="xreg + timesfm",
    )
    all_outputs_w2.extend(outputs_chunk)

elapsed = time.time() - start
print(f"Total time: {elapsed/60:.1f} minutes")

results_tfm_w2 = []
for i, (s, d) in enumerate(valid_pairs_w2):
    pred = all_outputs_w2[i][-len(actuals_batch_w2[i]):]
    for dt, actual_val, pred_val, is_hol in zip(dates_batch_w2[i], actuals_batch_w2[i], pred, holiday_test_batch_w2[i]):
        results_tfm_w2.append({'Store': s, 'Dept': d, 'Date': dt, 'Weekly_Sales': pred_val,
                                 'Actual': actual_val, 'IsHoliday': is_hol})

results_tfm_w2_df = pd.DataFrame(results_tfm_w2)
results_tfm_w2_df['Date'] = pd.to_datetime(results_tfm_w2_df['Date'])

wmae_w2_noblend = wmae(results_tfm_w2_df['Actual'], results_tfm_w2_df['Weekly_Sales'], results_tfm_w2_df['IsHoliday'])

blended_w2 = blend_holiday_predictions(
    results_tfm_w2_df[['Store','Dept','Date','Weekly_Sales']].copy(),
    df_clean, results_tfm_w2_df[['Store','Dept','Date','IsHoliday']], alpha=0.55
)
merged_w2 = blended_w2.merge(results_tfm_w2_df[['Store','Dept','Date','Actual','IsHoliday']], on=['Store','Dept','Date'])
wmae_w2_blended = wmae(merged_w2['Actual'], merged_w2['Weekly_Sales'], merged_w2['IsHoliday'])

with mlflow.start_run(run_name="TimesFM_FullScale_Window2_Validation"):
    mlflow.log_metric("wmae_no_blend", wmae_w2_noblend)
    mlflow.log_metric("wmae_with_blend", wmae_w2_blended)
    mlflow.log_metric("window1_wmae_blended", 1629.55)

print(f"Window 2 WMAE (no blend): {wmae_w2_noblend:.2f}")
print(f"Window 2 WMAE (with blend): {wmae_w2_blended:.2f}")
print(f"Window 1 WMAE (with blend): 1629.55")

# Error Breakdown by Holiday Type

In [ ]:
holiday_dates_map = {
    "2011-11-25": "Thanksgiving", "2011-12-30": "Christmas", "2012-02-10": "SuperBowl"
}
merged_final['holiday_name'] = merged_final['Date'].dt.strftime('%Y-%m-%d').map(holiday_dates_map).fillna('Regular')

breakdown_tfm = merged_final.groupby('holiday_name', observed=True).apply(
    lambda g: wmae(g['Actual'], g['Weekly_Sales'], g['IsHoliday']), include_groups=False)
print(breakdown_tfm)

# XReg Ridge Regularization Investigation

In [ ]:
ridge_values = [0.0, 1.0, 5.0, 10.0]
ridge_results_w1 = {}

for ridge_val in ridge_values:
    outputs_r, _ = model_true.forecast_with_covariates(
        inputs=inputs_list,
        dynamic_numerical_covariates={'is_holiday': holiday_covariates},
        static_categorical_covariates={'store_type': static_types},
        xreg_mode="xreg + timesfm",
        ridge=ridge_val,
    )
    scores_r = []
    for i, (s, d) in enumerate(examples):
        pred = outputs_r[i][-39:] if len(outputs_r[i]) > 39 else outputs_r[i]
        scores_r.append(wmae(actuals_list[i], pred, holidays_test_list[i]))
    avg_r = np.mean(scores_r)
    ridge_results_w1[ridge_val] = avg_r
    print(f"ridge={ridge_val}, Window 1: avg WMAE = {avg_r:.2f}")

In [ ]:
def get_holiday_covariate_w2(store, dept, context_len):
    hist = df_clean[(df_clean['Store']==store) & (df_clean['Dept']==dept)].sort_values('Date')
    train_part = hist[hist['Date'] < VALIDATION_START_2]
    test_part = hist[(hist['Date'] >= VALIDATION_START_2) & (hist['Date'] < VALIDATION_END_2)]
    hist_context = train_part.tail(context_len)
    combined = pd.concat([hist_context['IsHoliday'], test_part['IsHoliday']]).astype(float).tolist()
    return combined, train_part['Weekly_Sales'].values[-context_len:], test_part

ridge_results_w2 = {}
inputs_w2_sample, holiday_cov_w2_sample, static_w2_sample, actuals_w2_sample, holtest_w2_sample = [], [], [], [], []

for s, d in examples:
    cov, train_y, test_part = get_holiday_covariate_w2(s, d, 104)
    inputs_w2_sample.append(train_y)
    holiday_cov_w2_sample.append(cov)
    static_w2_sample.append(test_part['Type'].iloc[0] if len(test_part) else 'A')
    actuals_w2_sample.append(test_part['Weekly_Sales'].values)
    holtest_w2_sample.append(test_part['IsHoliday'].values)

for ridge_val in ridge_values:
    outputs_r2, _ = model_true.forecast_with_covariates(
        inputs=inputs_w2_sample,
        dynamic_numerical_covariates={'is_holiday': holiday_cov_w2_sample},
        static_categorical_covariates={'store_type': static_w2_sample},
        xreg_mode="xreg + timesfm",
        ridge=ridge_val,
    )
    scores_r2 = []
    for i in range(len(examples)):
        pred = outputs_r2[i][-len(actuals_w2_sample[i]):]
        scores_r2.append(wmae(actuals_w2_sample[i], pred, holtest_w2_sample[i]))
    avg_r2 = np.mean(scores_r2)
    ridge_results_w2[ridge_val] = avg_r2
    print(f"ridge={ridge_val}, Window 2: avg WMAE = {avg_r2:.2f}")

# XReg Mode Comparison (xreg+timesfm vs timesfm+xreg)

In [ ]:
mode_results = {}
for mode in ["xreg + timesfm", "timesfm + xreg"]:
    outputs_m, _ = model_true.forecast_with_covariates(
        inputs=inputs_list,
        dynamic_numerical_covariates={'is_holiday': holiday_covariates},
        static_categorical_covariates={'store_type': static_types},
        xreg_mode=mode,
    )
    scores_m = []
    for i, (s, d) in enumerate(examples):
        pred = outputs_m[i][-39:] if len(outputs_m[i]) > 39 else outputs_m[i]
        scores_m.append(wmae(actuals_list[i], pred, holidays_test_list[i]))
    avg_m = np.mean(scores_m)
    mode_results[mode] = avg_m

    with mlflow.start_run(run_name=f"TimesFM_XRegMode_{mode.replace(' ','_').replace('+','plus')}"):
        mlflow.log_param("xreg_mode", mode)
        mlflow.log_metric("avg_wmae", avg_m)

    print(f"xreg_mode='{mode}': avg WMAE = {avg_m:.2f}")

# Additional Covariates (CPI, Unemployment, Temperature, Size)

In [ ]:
def get_extra_covariates(store, dept, context_len):
    hist = df_clean[(df_clean['Store']==store) & (df_clean['Dept']==dept)].sort_values('Date')
    hist_context = hist.iloc[:-39].tail(context_len)
    future_part = hist.iloc[-39:]
    combined = pd.concat([hist_context, future_part])
    return {
        'cpi': combined['CPI'].tolist(),
        'unemployment': combined['Unemployment'].tolist(),
        'temperature': combined['Temperature'].tolist(),
    }

extra_cov_lists = {'cpi': [], 'unemployment': [], 'temperature': []}
size_static = []

for s, d in examples:
    cov = get_extra_covariates(s, d, 104)
    for k in extra_cov_lists:
        extra_cov_lists[k].append(cov[k])
    size_static.append(df_clean[(df_clean['Store']==s) & (df_clean['Dept']==d)]['Size'].iloc[0])

outputs_extra, _ = model_true.forecast_with_covariates(
    inputs=inputs_list,
    dynamic_numerical_covariates={'is_holiday': holiday_covariates, **extra_cov_lists},
    static_categorical_covariates={'store_type': static_types},
    static_numerical_covariates={'size': size_static},
    xreg_mode="xreg + timesfm",
)

scores_extra = []
for i, (s, d) in enumerate(examples):
    pred = outputs_extra[i][-39:] if len(outputs_extra[i]) > 39 else outputs_extra[i]
    scores_extra.append(wmae(actuals_list[i], pred, holidays_test_list[i]))

avg_extra = np.mean(scores_extra)
print(f"With extra covariates (CPI, Unemployment, Temperature, Size): avg WMAE = {avg_extra:.2f}")
print(f"With just IsHoliday, Store Type: 3807.36")

with mlflow.start_run(run_name="TimesFM_ExtraCovariates"):
    mlflow.log_param("covariates", "is_holiday, store_type, cpi, unemployment, temperature, size")
    mlflow.log_metric("avg_wmae", avg_extra)
    mlflow.log_metric("baseline_wmae", 3807.36)

In [ ]:
best_timesfm_config = {
    'max_context': 104,
    'covariates': ['is_holiday', 'store_type'],
    'xreg_mode': 'xreg + timesfm',
    'ridge': 0.0,
    'blend_alpha': 0.55,
}
print("Final TimesFM configuration locked in:", best_timesfm_config)

# Final Model: Full History, Real Test Predictions

In [ ]:
all_pairs_final = list(df_clean.groupby(['Store','Dept']).groups.keys())

valid_pairs_final = []
inputs_final = []
holiday_cov_final = []
static_final = []
dates_needed_final = []

for s, d in all_pairs_final:
    series = df_clean[(df_clean['Store'] == s) & (df_clean['Dept'] == d)].sort_values('Date')
    train_y = series['Weekly_Sales'].values[-104:]
    if len(train_y) < 10:
        continue

    test_dates = df_test[(df_test['Store']==s) & (df_test['Dept']==d)].sort_values('Date')
    if len(test_dates) == 0:
        continue

    hist_holiday = series['IsHoliday'].tail(len(train_y)).astype(float).tolist()
    future_holiday = test_dates['IsHoliday'].astype(float).tolist()

    valid_pairs_final.append((s, d))
    inputs_final.append(train_y)
    holiday_cov_final.append(hist_holiday + future_holiday)
    static_final.append(series['Type'].iloc[0])
    dates_needed_final.append(test_dates['Date'].values)

print(f"Series to predict: {len(valid_pairs_final)} / {len(all_pairs_final)}")

In [ ]:
all_outputs_final = []
start = time.time()

for i in range(0, len(inputs_final), BATCH_SIZE):
    chunk_inputs = inputs_final[i:i+BATCH_SIZE]
    chunk_holiday = holiday_cov_final[i:i+BATCH_SIZE]
    chunk_static = static_final[i:i+BATCH_SIZE]

    outputs_chunk, _ = model_true.forecast_with_covariates(
        inputs=chunk_inputs,
        dynamic_numerical_covariates={'is_holiday': chunk_holiday},
        static_categorical_covariates={'store_type': chunk_static},
        xreg_mode="xreg + timesfm",
        ridge=0.0,
    )
    all_outputs_final.extend(outputs_chunk)
    if (i // BATCH_SIZE) % 5 == 0:
        print(f"  {i + len(chunk_inputs)}/{len(inputs_final)} done...")

elapsed = time.time() - start
print(f"Total time: {elapsed/60:.1f} minutes")

In [ ]:
warm_results_final = []
for i, (s, d) in enumerate(valid_pairs_final):
    n = len(dates_needed_final[i])
    pred = all_outputs_final[i][-n:]
    for dt, val in zip(dates_needed_final[i], pred):
        warm_results_final.append({'Store': s, 'Dept': d, 'Date': dt, 'Weekly_Sales': val})

warm_df_final = pd.DataFrame(warm_results_final)
warm_df_final['Date'] = pd.to_datetime(warm_df_final['Date'])
print("Warm predictions:", warm_df_final.shape)

holiday_flags_final = df_test[['Store','Dept','Date','IsHoliday']].copy()
holiday_flags_final['Date'] = pd.to_datetime(holiday_flags_final['Date'])
warm_blended_final = blend_holiday_predictions(warm_df_final, df_clean, holiday_flags_final, alpha=0.55)

covered_final = set(zip(warm_blended_final.Store, warm_blended_final.Dept, warm_blended_final.Date))
df_test_dt = df_test.copy()
df_test_dt['Date'] = pd.to_datetime(df_test_dt['Date'])
df_test_dt['covered'] = df_test_dt.apply(lambda r: (r['Store'], r['Dept'], r['Date']) in covered_final, axis=1)
missing_final = df_test_dt[~df_test_dt['covered']].drop(columns=['covered'])
print("Genuinely missing rows to fill:", len(missing_final))

cold_result_final = coldstart_fallback(df_clean, missing_final)
if len(cold_result_final) < len(missing_final):
    recent = df_clean[df_clean['Date'] >= df_clean['Date'].max() - pd.Timedelta(weeks=52)]
    med = recent.groupby(['Type','Dept'])['Weekly_Sales'].median()
    missing_final = missing_final.merge(df_clean[['Store','Type']].drop_duplicates(), on='Store', how='left')
    missing_final['Weekly_Sales'] = [med.get((t,d2), 0.0) for t,d2 in zip(missing_final['Type'], missing_final['Dept'])]
    cold_result_final = missing_final[['Store','Dept','Date','Weekly_Sales']]
else:
    cold_result_final = cold_result_final[['Store','Dept','Date','y_fallback']].rename(columns={'y_fallback':'Weekly_Sales'})

final_predictions_tfm = pd.concat([
    warm_blended_final[['Store','Dept','Date','Weekly_Sales']],
    cold_result_final[['Store','Dept','Date','Weekly_Sales']]
], ignore_index=True)
final_predictions_tfm['Weekly_Sales'] = final_predictions_tfm['Weekly_Sales'].clip(lower=0)
final_predictions_tfm['Id'] = (final_predictions_tfm['Store'].astype(str) + '_' +
                                 final_predictions_tfm['Dept'].astype(str) + '_' +
                                 final_predictions_tfm['Date'].dt.strftime('%Y-%m-%d'))

submission_tfm = final_predictions_tfm[['Id','Weekly_Sales']].sort_values('Id').reset_index(drop=True)
submission_tfm = submission_tfm.drop_duplicates(subset='Id', keep='first')
assert len(submission_tfm) == len(test), f"Row mismatch: {len(submission_tfm)} vs {len(test)}"

submission_tfm.to_csv('submission_timesfm.csv', index=False)
print(submission_tfm.shape)
print(submission_tfm.head())

In [ ]:
print(missing_final.columns.tolist())
print(missing_final.shape)

In [ ]:
missing_final['Type'] = missing_final['Type_x']

recent = df_clean[df_clean['Date'] >= df_clean['Date'].max() - pd.Timedelta(weeks=52)]
med = recent.groupby(['Type','Dept'])['Weekly_Sales'].median()
missing_final['Weekly_Sales'] = [med.get((t,d2), 0.0) for t,d2 in zip(missing_final['Type'], missing_final['Dept'])]
missing_final['Weekly_Sales'] = missing_final['Weekly_Sales'].clip(lower=0)
cold_result_final = missing_final[['Store','Dept','Date','Weekly_Sales']]

final_predictions_tfm = pd.concat([
    warm_blended_final[['Store','Dept','Date','Weekly_Sales']],
    cold_result_final[['Store','Dept','Date','Weekly_Sales']]
], ignore_index=True)
final_predictions_tfm['Weekly_Sales'] = final_predictions_tfm['Weekly_Sales'].clip(lower=0)
final_predictions_tfm['Id'] = (final_predictions_tfm['Store'].astype(str) + '_' +
                                 final_predictions_tfm['Dept'].astype(str) + '_' +
                                 final_predictions_tfm['Date'].dt.strftime('%Y-%m-%d'))

submission_tfm = final_predictions_tfm[['Id','Weekly_Sales']].sort_values('Id').reset_index(drop=True)
submission_tfm = submission_tfm.drop_duplicates(subset='Id', keep='first')
assert len(submission_tfm) == len(test), f"Row mismatch: {len(submission_tfm)} vs {len(test)}"

submission_tfm.to_csv('submission_timesfm.csv', index=False)
print(submission_tfm.shape)
print(submission_tfm.head())

# Pipeline Artifact (with reload verification from the start)

In [ ]:
class TimesFMPipeline(mlflow.pyfunc.PythonModel):
    def __init__(self, train_clean, config, batch_size=200):
        self.train_clean = train_clean
        self.config = config
        self.batch_size = batch_size

    def predict(self, context, model_input):
        test_merged = load_and_merge(model_input.copy(), features, stores)
        test_clean = preprocess(test_merged)
        all_pairs = list(self.train_clean.groupby(['Store','Dept']).groups.keys())
        valid_pairs_p, inputs_p, holiday_cov_p, static_p, dates_p = [], [], [], [], []
        for s, d in all_pairs:
            series = self.train_clean[(self.train_clean['Store']==s) & (self.train_clean['Dept']==d)].sort_values('Date')
            train_y = series['Weekly_Sales'].values[-104:]
            if len(train_y) < 10:
                continue
            test_dates = test_clean[(test_clean['Store']==s) & (test_clean['Dept']==d)].sort_values('Date')
            if len(test_dates) == 0:
                continue
            hist_holiday = series['IsHoliday'].tail(len(train_y)).astype(float).tolist()
            future_holiday = test_dates['IsHoliday'].astype(float).tolist()
            valid_pairs_p.append((s, d))
            inputs_p.append(train_y)
            holiday_cov_p.append(hist_holiday + future_holiday)
            static_p.append(series['Type'].iloc[0])
            dates_p.append(test_dates['Date'].values)

        outputs_p = []
        for i in range(0, len(inputs_p), self.batch_size):
            chunk_out, _ = self.config['model'].forecast_with_covariates(
                inputs=inputs_p[i:i+self.batch_size],
                dynamic_numerical_covariates={'is_holiday': holiday_cov_p[i:i+self.batch_size]},
                static_categorical_covariates={'store_type': static_p[i:i+self.batch_size]},
                xreg_mode="xreg + timesfm",
                ridge=0.0,
            )
            outputs_p.extend(chunk_out)

        results_p = []
        for i, (s, d) in enumerate(valid_pairs_p):
            n = len(dates_p[i])
            pred = outputs_p[i][-n:]
            for dt, val in zip(dates_p[i], pred):
                results_p.append({'Store': s, 'Dept': d, 'Date': dt, 'Weekly_Sales': val})
        warm_p = pd.DataFrame(results_p)
        warm_p['Date'] = pd.to_datetime(warm_p['Date'])
        holiday_flags_p = test_clean[['Store','Dept','Date','IsHoliday']].copy()
        warm_blended_p = blend_holiday_predictions(warm_p, self.train_clean, holiday_flags_p, alpha=0.55)
        covered_p = set(zip(warm_blended_p.Store, warm_blended_p.Dept, warm_blended_p.Date))
        test_clean_c = test_clean.copy()
        test_clean_c['covered'] = test_clean_c.apply(lambda r: (r['Store'], r['Dept'], r['Date']) in covered_p, axis=1)
        missing_p = test_clean_c[~test_clean_c['covered']].drop(columns=['covered'])
        recent = self.train_clean[self.train_clean['Date'] >= self.train_clean['Date'].max() - pd.Timedelta(weeks=52)]
        med = recent.groupby(['Type','Dept'])['Weekly_Sales'].median()
        missing_p = missing_p.copy()
        missing_p['Weekly_Sales'] = [med.get((t,d2), 0.0) for t,d2 in zip(missing_p['Type'], missing_p['Dept'])]
        final = pd.concat([warm_blended_p[['Store','Dept','Date','Weekly_Sales']],
                            missing_p[['Store','Dept','Date','Weekly_Sales']]], ignore_index=True)
        final['Weekly_Sales'] = final['Weekly_Sales'].clip(lower=0)
        return final

In [ ]:
pipeline_tfm = TimesFMPipeline(train_clean=df_clean, config={'model': model_true}, batch_size=200)

In [ ]:
with mlflow.start_run(run_name="TimesFM_Pipeline_Wrapped"):
    mlflow.log_params({k: str(v) for k, v in best_timesfm_config.items()})
    mlflow.log_param("scope", "full dataset, all series with >=10 weeks history, batch_size=200")
    model_info = mlflow.pyfunc.log_model(python_model=pipeline_tfm, name="pipeline")

loaded_model_tfm = mlflow.pyfunc.load_model(model_info.model_uri)
reloaded_preds_tfm = loaded_model_tfm.predict(test)

reloaded_preds_tfm['Id'] = (reloaded_preds_tfm['Store'].astype(str) + '_' +
                              reloaded_preds_tfm['Dept'].astype(str) + '_' +
                              reloaded_preds_tfm['Date'].dt.strftime('%Y-%m-%d'))
reloaded_submission_tfm = reloaded_preds_tfm[['Id','Weekly_Sales']].sort_values('Id').reset_index(drop=True)
reloaded_submission_tfm = reloaded_submission_tfm.drop_duplicates(subset='Id', keep='first')

old_submission_tfm = pd.read_csv('submission_timesfm.csv')
comparison_tfm = reloaded_submission_tfm.merge(old_submission_tfm, on='Id', suffixes=('_reloaded','_original'))
max_diff_tfm = (comparison_tfm['Weekly_Sales_reloaded'] - comparison_tfm['Weekly_Sales_original']).abs().max()

print("Rows compared:", len(comparison_tfm), "/", len(old_submission_tfm))
print("Max diff (reloaded model vs original submission):", max_diff_tfm)

In [ ]:
pipeline_tfm = TimesFMPipeline(train_clean=df_clean, config={'model': model_true})

demo_test_tfm = test[test.apply(lambda r: (r['Store'], r['Dept']) in set(examples), axis=1)].copy()
demo_result_tfm = pipeline_tfm.predict(None, demo_test_tfm)
print(demo_result_tfm.shape)
print(demo_result_tfm.head())

In [ ]:
with mlflow.start_run(run_name="TimesFM_Pipeline_Wrapped"):
    mlflow.log_params({k: str(v) for k, v in best_timesfm_config.items()})
    mlflow.log_param("scope", "full dataset, all series with >=10 weeks history")
    model_info = mlflow.pyfunc.log_model(python_model=pipeline_tfm, name="pipeline")

print("Pipeline logged. Now reloading and verifying (this will take a while)...")

loaded_model_tfm = mlflow.pyfunc.load_model(model_info.model_uri)
reloaded_preds_tfm = loaded_model_tfm.predict(test)

reloaded_preds_tfm['Id'] = (reloaded_preds_tfm['Store'].astype(str) + '_' +
                              reloaded_preds_tfm['Dept'].astype(str) + '_' +
                              reloaded_preds_tfm['Date'].dt.strftime('%Y-%m-%d'))
reloaded_submission_tfm = reloaded_preds_tfm[['Id','Weekly_Sales']].sort_values('Id').reset_index(drop=True)
reloaded_submission_tfm = reloaded_submission_tfm.drop_duplicates(subset='Id', keep='first')

old_submission_tfm = pd.read_csv('submission_timesfm.csv')
comparison_tfm = reloaded_submission_tfm.merge(old_submission_tfm, on='Id', suffixes=('_reloaded','_original'))
max_diff_tfm = (comparison_tfm['Weekly_Sales_reloaded'] - comparison_tfm['Weekly_Sales_original']).abs().max()

print("Rows compared:", len(comparison_tfm), "/", len(old_submission_tfm))
print("Max diff (reloaded model vs original submission):", max_diff_tfm)

In [ ]:
# Single call, all 5 at once
outputs_single, _ = model_true.forecast_with_covariates(
    inputs=inputs_list,
    dynamic_numerical_covariates={'is_holiday': holiday_covariates},
    static_categorical_covariates={'store_type': static_types},
    xreg_mode="xreg + timesfm", ridge=0.0,
)

padded_inputs = inputs_list + [inputs_list[0]] * 195
padded_holiday = holiday_covariates + [holiday_covariates[0]] * 195
padded_static = static_types + [static_types[0]] * 195

outputs_batched, _ = model_true.forecast_with_covariates(
    inputs=padded_inputs,
    dynamic_numerical_covariates={'is_holiday': padded_holiday},
    static_categorical_covariates={'store_type': padded_static},
    xreg_mode="xreg + timesfm", ridge=0.0,
)

for i in range(5):
    diff = np.max(np.abs(np.array(outputs_single[i]) - np.array(outputs_batched[i])))
    print(f"Series {i}: max diff between single-call and batched-call = {diff}")

In [ ]:
# Reconstruct the exact first batch of 200 from your ORIGINAL full run, same order
first_200_pairs = valid_pairs_final[:200]

first_200_inputs = inputs_final[:200]
first_200_holiday = holiday_cov_final[:200]
first_200_static = static_final[:200]
first_200_dates = dates_needed_final[:200]

outputs_check, _ = model_true.forecast_with_covariates(
    inputs=first_200_inputs,
    dynamic_numerical_covariates={'is_holiday': first_200_holiday},
    static_categorical_covariates={'store_type': first_200_static},
    xreg_mode="xreg + timesfm", ridge=0.0,
)

# Build a small results frame just for these 200
check_results = []
for i, (s, d) in enumerate(first_200_pairs):
    n = len(first_200_dates[i])
    pred = outputs_check[i][-n:]
    for dt, val in zip(first_200_dates[i], pred):
        check_results.append({'Store': s, 'Dept': d, 'Date': pd.to_datetime(dt), 'Weekly_Sales_check': val})

check_df = pd.DataFrame(check_results)
check_df['Id'] = (check_df['Store'].astype(str) + '_' + check_df['Dept'].astype(str) + '_' + check_df['Date'].dt.strftime('%Y-%m-%d'))

original = pd.read_csv('submission_timesfm.csv')
compare = check_df.merge(original, on='Id')
max_diff_check = (compare['Weekly_Sales_check'] - compare['Weekly_Sales']).abs().max()

print("Rows checked:", len(compare))
print("Max diff (first-200-batch recomputed vs original submission):", max_diff_check)

In [ ]:
s, d = valid_pairs_final[0]
print(f"Checking Store {s}, Dept {d}")

series = df_clean[(df_clean['Store'] == s) & (df_clean['Dept'] == d)].sort_values('Date')
train_y_check = series['Weekly_Sales'].values[-104:]
print("Train_y matches inputs_final[0]?", np.array_equal(train_y_check, inputs_final[0]))

test_dates_check = df_test[(df_test['Store']==s) & (df_test['Dept']==d)].sort_values('Date')
print("Number of test dates for this series:", len(test_dates_check))
print("dates_needed_final[0] length:", len(dates_needed_final[0]))

# Single-series forecast, right now, fresh
single_out, _ = model_true.forecast_with_covariates(
    inputs=[train_y_check],
    dynamic_numerical_covariates={'is_holiday': [holiday_cov_final[0]]},
    static_categorical_covariates={'store_type': [static_final[0]]},
    xreg_mode="xreg + timesfm", ridge=0.0,
)

pred_now = single_out[0][-len(test_dates_check):]

# Pull the ORIGINAL submission's values for this exact series
ids_this_series = (str(s) + '_' + str(d) + '_' + test_dates_check['Date'].astype(str))
original = pd.read_csv('submission_timesfm.csv')
orig_vals = original[original['Id'].isin(ids_this_series)].sort_values('Id')['Weekly_Sales'].values

print()
print("Freshly computed (no blend):", pred_now[:5])
print("Original submission (blended):", orig_vals[:5])

In [ ]:
print("Duplicates in check_df Id:", check_df['Id'].duplicated().sum())
print("Duplicates in original Id (within matched rows):", compare['Id'].duplicated().sum())
print()
print("Rows in check_df:", len(check_df))
print("Rows in compare (after merge):", len(compare))
print()
worst = compare.loc[(compare['Weekly_Sales_check'] - compare['Weekly_Sales']).abs().idxmax()]
print(worst)

In [ ]:
first_200_pairs = valid_pairs_final[:200]
first_200_inputs = inputs_final[:200]
first_200_holiday = holiday_cov_final[:200]
first_200_static = static_final[:200]
first_200_dates = dates_needed_final[:200]

outputs_check2, _ = model_true.forecast_with_covariates(
    inputs=first_200_inputs,
    dynamic_numerical_covariates={'is_holiday': first_200_holiday},
    static_categorical_covariates={'store_type': first_200_static},
    xreg_mode="xreg + timesfm", ridge=0.0,
)

check_results2 = []
for i, (s, d) in enumerate(first_200_pairs):
    n = len(first_200_dates[i])
    pred = outputs_check2[i][-n:]
    for dt, val in zip(first_200_dates[i], pred):
        check_results2.append({'Store': s, 'Dept': d, 'Date': pd.to_datetime(dt), 'Weekly_Sales': val})

check_df2 = pd.DataFrame(check_results2)
holiday_flags_check = df_test[['Store','Dept','Date','IsHoliday']].copy()
holiday_flags_check['Date'] = pd.to_datetime(holiday_flags_check['Date'])
check_blended = blend_holiday_predictions(check_df2, df_clean, holiday_flags_check, alpha=0.55)
check_blended['Id'] = (check_blended['Store'].astype(str) + '_' + check_blended['Dept'].astype(str) + '_' + check_blended['Date'].dt.strftime('%Y-%m-%d'))

original = pd.read_csv('submission_timesfm.csv')
compare2 = check_blended.merge(original, on='Id', suffixes=('_check','_orig'))
max_diff_check2 = (compare2['Weekly_Sales_check'] - compare2['Weekly_Sales_orig']).abs().max()

print("Rows checked:", len(compare2))
print("Max diff (blended vs blended, same batch of 200):", max_diff_check2)

In [ ]:
outputs_run1, _ = model_true.forecast_with_covariates(
    inputs=first_200_inputs,
    dynamic_numerical_covariates={'is_holiday': first_200_holiday},
    static_categorical_covariates={'store_type': first_200_static},
    xreg_mode="xreg + timesfm", ridge=0.0,
)

outputs_run2, _ = model_true.forecast_with_covariates(
    inputs=first_200_inputs,
    dynamic_numerical_covariates={'is_holiday': first_200_holiday},
    static_categorical_covariates={'store_type': first_200_static},
    xreg_mode="xreg + timesfm", ridge=0.0,
)

max_diff_same_session = max(np.max(np.abs(np.array(outputs_run1[i]) - np.array(outputs_run2[i]))) for i in range(len(outputs_run1)))
print("Max diff between two identical calls, same session, back to back:", max_diff_same_session)

In [ ]:
print(model_true.forecast_config)

In [ ]:
model_true = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
model_true.compile(timesfm.ForecastConfig(
    max_context=104,
    max_horizon=39,
    normalize_inputs=True,
    use_continuous_quantile_head=True,
    force_flip_invariance=True,
    infer_is_positive=True,
    fix_quantile_crossing=True,
    return_backcast=True,
))
print(model_true.forecast_config)

# Recovery cell

In [3]:
!pip install timesfm mlflow --quiet
import pandas as pd, numpy as np, mlflow, os, time
import matplotlib.pyplot as plt
import timesfm
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
dagshub_token = user_secrets.get_secret("DAGSHUB_TOKEN")
os.environ['MLFLOW_TRACKING_USERNAME'] = 'gdzag22'
os.environ['MLFLOW_TRACKING_PASSWORD'] = dagshub_token
mlflow.set_tracking_uri("https://dagshub.com/Nestor-Dzadzamia/walmart-sales-forecasting.mlflow")
mlflow.set_experiment("TimesFM_Training")

def load_and_merge(df, features, stores):
    out = df.merge(features, on=['Store', 'Date', 'IsHoliday'], how='left')
    out = out.merge(stores, on='Store', how='left')
    out['Date'] = pd.to_datetime(out['Date'])
    return out.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

MD_COLS = [f"MarkDown{i}" for i in range(1, 6)]

def preprocess(df):
    out = df.copy()
    out[MD_COLS] = out[MD_COLS].fillna(0)
    out[["CPI", "Unemployment"]] = out.groupby("Store")[["CPI", "Unemployment"]].ffill()
    return out

def wmae(y_true, y_pred, is_holiday):
    w = np.where(is_holiday, 5, 1)
    return np.sum(w * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(w)

def coldstart_fallback(df_train, df_test):
    train_pairs = set(zip(df_train.Store, df_train.Dept))
    mask = ~pd.Series(list(zip(df_test.Store, df_test.Dept)), index=df_test.index).isin(train_pairs)
    cold = df_test[mask].copy()
    recent = df_train[df_train['Date'] >= df_train['Date'].max() - pd.Timedelta(weeks=52)]
    med = recent.groupby(['Type', 'Dept'])['Weekly_Sales'].median()
    cold['y_fallback'] = [med.get((t, d), 0.0) for t, d in zip(cold['Type'], cold['Dept'])]
    cold['y_fallback'] = cold['y_fallback'].clip(lower=0)
    return cold

def get_seasonal_naive(dates_df, history_df):
    lookup_df = dates_df[['Store','Dept','Date']].copy()
    lookup_df['lookup_date'] = lookup_df['Date'] - pd.Timedelta(weeks=52)
    merged = lookup_df.merge(history_df[['Store','Dept','Date','Weekly_Sales']].rename(
        columns={'Date':'lookup_date','Weekly_Sales':'seasonal_naive'}), on=['Store','Dept','lookup_date'], how='left')
    return merged['seasonal_naive'].values

def blend_holiday_predictions(preds_df, history_df, holiday_flags_df, alpha=0.5):
    out = preds_df.merge(holiday_flags_df[['Store','Dept','Date','IsHoliday']], on=['Store','Dept','Date'], how='left')
    out['seasonal_naive'] = get_seasonal_naive(out, history_df)
    mask = out['IsHoliday'].fillna(False) & out['seasonal_naive'].notna()
    out.loc[mask, 'Weekly_Sales'] = alpha * out.loc[mask, 'seasonal_naive'] + (1 - alpha) * out.loc[mask, 'Weekly_Sales']
    return out.drop(columns=['IsHoliday', 'seasonal_naive'])

path = "/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/"
train = pd.read_csv(path + "train.csv.zip")
test = pd.read_csv(path + "test.csv.zip")
features = pd.read_csv(path + "features.csv.zip")
stores = pd.read_csv(path + "stores.csv")

df = load_and_merge(train, features, stores)
df_test = load_and_merge(test, features, stores)
df_clean = preprocess(df)

VALIDATION_START = df_test['Date'].min() - pd.Timedelta(weeks=52)
VALIDATION_END = VALIDATION_START + pd.Timedelta(weeks=39)

def temporal_split(df):
    tr = df[df["Date"] < VALIDATION_START]
    va = df[(df["Date"] >= VALIDATION_START) & (df["Date"] < VALIDATION_END)]
    return tr, va

tr, va = temporal_split(df_clean)

model_true = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
model_true.compile(timesfm.ForecastConfig(
    max_context=104, max_horizon=39, normalize_inputs=True,
    use_continuous_quantile_head=True, force_flip_invariance=True,
    infer_is_positive=True, fix_quantile_crossing=True,
    return_backcast=True,
))

BATCH_SIZE = 200
print("Recovery complete. tr:", tr.shape, "| va:", va.shape)

/usr/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


Recovery complete. tr: (267184, 16) | va: (115856, 16)


In [2]:
all_pairs_tfm = list(df_clean.groupby(['Store','Dept']).groups.keys())

valid_pairs = []
inputs_batch = []
holiday_cov_batch = []
static_type_batch = []
actuals_batch = []
holiday_test_batch = []
dates_batch = []

for s, d in all_pairs_tfm:
    tr_series = tr[(tr['Store'] == s) & (tr['Dept'] == d)].sort_values('Date')
    va_series = va[(va['Store'] == s) & (va['Dept'] == d)].sort_values('Date')

    if len(tr_series) < 10 or len(va_series) == 0:
        continue

    train_y = tr_series['Weekly_Sales'].values[-104:]
    test_y = va_series['Weekly_Sales'].values

    hist_holiday = tr_series['IsHoliday'].tail(len(train_y)).astype(float).tolist()
    future_holiday = va_series['IsHoliday'].astype(float).tolist()

    valid_pairs.append((s, d))
    inputs_batch.append(train_y)
    holiday_cov_batch.append(hist_holiday + future_holiday)
    static_type_batch.append(tr_series['Type'].iloc[0])
    actuals_batch.append(test_y)
    holiday_test_batch.append(va_series['IsHoliday'].values)
    dates_batch.append(va_series['Date'].values)

print(f"Valid series: {len(valid_pairs)} / {len(all_pairs_tfm)}")

all_outputs = []
start = time.time()
for i in range(0, len(inputs_batch), BATCH_SIZE):
    chunk_out, _ = model_true.forecast_with_covariates(
        inputs=inputs_batch[i:i+BATCH_SIZE],
        dynamic_numerical_covariates={'is_holiday': holiday_cov_batch[i:i+BATCH_SIZE]},
        static_categorical_covariates={'store_type': static_type_batch[i:i+BATCH_SIZE]},
        xreg_mode="xreg + timesfm", ridge=0.0,
    )
    all_outputs.extend(chunk_out)
elapsed = time.time() - start
print(f"Total time: {elapsed/60:.1f} minutes")

results_tfm = []
for i, (s, d) in enumerate(valid_pairs):
    pred = all_outputs[i][-len(actuals_batch[i]):]
    for dt, actual_val, pred_val, is_hol in zip(dates_batch[i], actuals_batch[i], pred, holiday_test_batch[i]):
        results_tfm.append({'Store': s, 'Dept': d, 'Date': dt, 'Weekly_Sales': pred_val,
                             'Actual': actual_val, 'IsHoliday': is_hol})

results_tfm_df = pd.DataFrame(results_tfm)
results_tfm_df['Date'] = pd.to_datetime(results_tfm_df['Date'])

overall_wmae_tfm = wmae(results_tfm_df['Actual'], results_tfm_df['Weekly_Sales'], results_tfm_df['IsHoliday'])
print(f"CORRECTED Full-scale WMAE (Window 1, shared tr/va split, no blend): {overall_wmae_tfm:.2f}")

blended_tfm = blend_holiday_predictions(
    results_tfm_df[['Store','Dept','Date','Weekly_Sales']].copy(),
    tr, results_tfm_df[['Store','Dept','Date','IsHoliday']], alpha=0.55
)
merged_final = blended_tfm.merge(results_tfm_df[['Store','Dept','Date','Actual','IsHoliday']], on=['Store','Dept','Date'])
overall_wmae_tfm_blended = wmae(merged_final['Actual'], merged_final['Weekly_Sales'], merged_final['IsHoliday'])
print(f"CORRECTED Full-scale WMAE (Window 1, shared tr/va split, with blend): {overall_wmae_tfm_blended:.2f}")

with mlflow.start_run(run_name="TimesFM_FullScale_Validation_CORRECTED"):
    mlflow.log_param("fix", "context from tr, evaluated against va (shared split), replaces invalid last-39-raw-weeks version")
    mlflow.log_metric("wmae_no_blend", overall_wmae_tfm)
    mlflow.log_metric("wmae_with_blend", overall_wmae_tfm_blended)
    mlflow.log_metric("previous_invalid_wmae", 1629.55)

Valid series: 3067 / 3331
Total time: 13.3 minutes
CORRECTED Full-scale WMAE (Window 1, shared tr/va split, no blend): 2699.26
CORRECTED Full-scale WMAE (Window 1, shared tr/va split, with blend): 2251.43
🏃 View run TimesFM_FullScale_Validation_CORRECTED at: https://dagshub.com/Nestor-Dzadzamia/walmart-sales-forecasting.mlflow/#/experiments/9/runs/00a5739d94c348b583dc5e5089448a6c
🧪 View experiment at: https://dagshub.com/Nestor-Dzadzamia/walmart-sales-forecasting.mlflow/#/experiments/9
